In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window


@dp.materialized_view(
    name="shipments_batch_silver",
    comment="Latest batch enrichment record per shipment"
)
def shipments_batch_silver():

    window = (
        Window
        .partitionBy("shipment_id")
        .orderBy(F.col("batch_created_ts").desc())
    )

    return (
        spark.read.table("shipments_batch_bronze")
        .select(
            "shipment_id",
            F.upper(F.trim("service_level")).alias("service_level"),
            F.upper(F.trim("equipment_type")).alias("equipment_type"),
            F.trim("origin_terminal").alias("origin_terminal"),
            F.trim("destination_terminal").alias("destination_terminal"),
            "weight_lbs",
            "batch_created_ts"
        )
        .filter(F.col("shipment_id").isNotNull())
        .withColumn("_row_num", F.row_number().over(window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )